In [55]:
from transformers import AutoTokenizer, AutoModel, AutoModelForTokenClassification, TrainingArguments, Trainer, pipeline
import platform
import torch
import numpy as np
import pandas as pd
import re
from collections import defaultdict
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from typing import Any, Dict, List, Optional, Union
import torch.nn as nn
from torch.utils.data import Subset
from sklearn.metrics import f1_score

In [56]:
def parse_multi_report_pubtator(raw_data):
    # If the input is bytes, decode it to a string first
    if isinstance(raw_data, bytes):
        raw_data = raw_data.decode('utf-8')

    report_groups = defaultdict(lambda: {'title': '', 'abstract': '', 'annotations': []})
    lines = raw_data.strip().split('\n')
    
    # 1. Grouping Phase: Separate data by PMID
    for line in lines:
        if not line.strip(): 
            continue
        
        parts = line.split('|')
        if len(parts) > 2:
            pmid, type_flag, content = parts[0], parts[1], parts[2]
            if type_flag == 't':
                report_groups[pmid]['title'] = content
            elif type_flag == 'a':
                report_groups[pmid]['abstract'] = content
        else:
            tab_parts = line.split('\t')
            if len(tab_parts) >= 5:
                pmid = tab_parts[0]
                report_groups[pmid]['annotations'].append({
                    'start': int(tab_parts[1]),
                    'end': int(tab_parts[2]),
                    'tags': tab_parts[4].split(',')
                })

    all_results = []

    # 2. Processing Phase: Handle each report independently
    for pmid, data in report_groups.items():
        # Using a space to separate title and abstract to maintain offset integrity
        full_text = data['title'] + " " + data['abstract']
        
        report_words = []
        report_tags = []
        
        for match in re.finditer(r'\w+', full_text):
            start, end = match.start(), match.end()
            word = match.group()
            
            current_word_tags = set()
            for ann in data['annotations']:
                if start >= ann['start'] and end <= ann['end']:
                    for t in ann['tags']:
                        current_word_tags.add(t)
            
            report_words.append(word)
            
            # --- Logic Update Start ---
            # If the set is empty, we tag it with 'O'
            if not current_word_tags:
                report_tags.append(['O'])
            else:
                report_tags.append(sorted(list(current_word_tags)))
            # --- Logic Update End ---
        
        all_results.append({
            'pmid': pmid,
            'words': report_words,
            'tags': report_tags
        })
        
    return all_results

In [57]:
@dataclass
class DataCollatorForMultiLabelTokenClassification:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        labels = [feature.pop("labels") for feature in features]
        
        # Standard padding for input_ids, attention_mask, etc.
        batch = self.tokenizer.pad(
            features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        
        # Manual padding for the multi-hot label vectors
        sequence_length = batch["input_ids"].shape[1]
        num_labels = len(labels[0][0])
        batch_size = len(labels)
        
        # Create a padded tensor of zeros for labels
        padded_labels = torch.zeros((batch_size, sequence_length, num_labels))
        
        for i, label_set in enumerate(labels):
            # Truncate if the labels are longer than the padded sequence
            end = min(len(label_set), sequence_length)
            padded_labels[i, :end, :] = torch.tensor(label_set[:end])
            
        batch["labels"] = padded_labels
        return batch

In [58]:
with open("metadata/corpus_pubtator_pmids_train.txt") as f:
    train_text = [line.strip() for line in f if line.strip()]
with open("metadata/corpus_pubtator_pmids_test.txt") as f:
    test_text = [line.strip() for line in f if line.strip()]
with open("metadata/corpus_pubtator_pmids_dev.txt") as f:
    dev_text = [line.strip() for line in f if line.strip()]
with open("metadata/tags.txt") as f:
    label_list = [line.split(',')[0] for line in f.read().strip().split('\n')]

In [59]:
# Loading the full text
with open("metadata/corpus_pubtator.txt", "r") as f:
    full_text = f.read()

results = parse_multi_report_pubtator(full_text)
corpus_id_dict = {item['pmid']: item for item in results}
train_results = []
dev_results = []
test_results = []

for pmid in corpus_id_dict:
    if pmid in train_text:
        train_results.append(corpus_id_dict[pmid])
    elif pmid in dev_text:
        dev_results.append(corpus_id_dict[pmid])
    elif pmid in test_text:
        test_results.append(corpus_id_dict[pmid])
    else:
        pass


In [60]:
def align_labels_with_tokens(tokenizer, words, tags, label_to_id):
    tokenized_inputs = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=512,
        padding='max_length'
    )

    word_ids = tokenized_inputs.word_ids()
    new_labels = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            # [CLS], [SEP], padding
            new_labels.append([0.0] * len(label_to_id))

        elif word_idx != previous_word_idx:
            # First sub-token of a word — assign real label
            multi_hot = [0.0] * len(label_to_id)
            current_tags = tags[word_idx]
            if current_tags != ['O']:
                for tag in current_tags:
                    if tag in label_to_id:
                        multi_hot[label_to_id[tag]] = 1.0
            new_labels.append(multi_hot)

        else:
        
            new_labels.append([-1.0] * len(label_to_id))

        previous_word_idx = word_idx  

    return tokenized_inputs, new_labels


class AlignPubTator(torch.utils.data.Dataset):
    def __init__(self, parsed_data, tokenizer, label2id):
        self.data = parsed_data
        self.tokenizer = tokenizer
        self.label2id = label2id

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        inputs, labels = align_labels_with_tokens(
            self.tokenizer, 
            item['words'], 
            item['tags'], 
            self.label2id
        )
        
        # Remove the batch dimension added by tokenizer() call inside the function
        # and ensure everything is a list for the data collator to handle
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": labels
        }

In [61]:
def compute_metrics_crf(eval_pred):
    """
    eval_pred.predictions : [b, T, L, 3]  emission logits
                            (may arrive as a tuple when Trainer collects
                             outputs from _CRFOutput — we unwrap if needed)
    eval_pred.label_ids   : [b, T, L]     float labels (-1 / 0 / 1)
    """
    logits, labels = eval_pred

    # Trainer sometimes wraps outputs in a tuple; unwrap to get the 4-D tensor
    if isinstance(logits, tuple):
        # logits[0] is typically the emission tensor [b, T, L, 3]
        logits = logits[0]

    # Safety: if still not 4-D (e.g. already [b,T,L]) skip the argmax step
    if logits.ndim == 4:
        pred_states = np.argmax(logits, axis=-1)  # [b, T, L]
    else:
        pred_states = logits.astype(int)          # already decoded

    # B or I → predicted positive; O → negative
    preds_bin = (pred_states > 0).astype(int)

    # Flatten
    preds_flat  = preds_bin.reshape(-1, preds_bin.shape[-1])
    labels_flat = labels.reshape(-1, labels.shape[-1])

    # Valid mask
    valid = labels_flat[:, 0] != -1.0
    preds_flat  = preds_flat[valid]
    labels_flat = labels_flat[valid].astype(int)

    return {
        "f1_micro": f1_score(labels_flat, preds_flat, average="micro", zero_division=0),
        "f1_macro": f1_score(labels_flat, preds_flat, average="macro", zero_division=0),
    }

In [62]:
class BinaryCRFLayer(nn.Module):
    """
    Vectorised binary CRF: runs `num_labels` independent B/I/O CRFs in
    parallel.  The three states are:
        0 = O  (outside)
        1 = B  (beginning)   — treated identically to I for multi-token
        2 = I  (inside)      entities; you can collapse B/I if preferred.
    """

    NUM_STATES = 3  # O, B, I

    def __init__(self, num_labels: int):
        super().__init__()
        self.num_labels = num_labels
        # Learned transition parameters: [num_labels, num_states, num_states]
        self.transitions = nn.Parameter(
            torch.randn(num_labels, self.NUM_STATES, self.NUM_STATES) * 0.1
        )
        # Start / end constraints (large negative = forbidden)
        self.start_transitions = nn.Parameter(
            torch.randn(num_labels, self.NUM_STATES) * 0.1
        )
        self.end_transitions = nn.Parameter(
            torch.randn(num_labels, self.NUM_STATES) * 0.1
        )

    def _forward_alg(self, emissions: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        batch, seq_len, L, S = emissions.shape
        assert L == self.num_labels and S == self.NUM_STATES

        alpha = self.start_transitions.unsqueeze(0) + emissions[:, 0]  # [b, L, S]

        for t in range(1, seq_len):
            emit_t = emissions[:, t].unsqueeze(-2)          # [b, L, 1, S] destination states.
            trans  = self.transitions.unsqueeze(0)           # [1, L, S, S] [from, to]
            score = alpha.unsqueeze(-1) + trans + emit_t    # [b, L, S, S]
            new_alpha = torch.logsumexp(score, dim=2)        # [b, L, S]

            mask_t = mask[:, t].view(batch, 1, 1)            # [b, 1, 1]
            alpha = torch.where(mask_t.bool(), new_alpha, alpha)

        alpha = alpha + self.end_transitions.unsqueeze(0)    # [b, L, S]
        return torch.logsumexp(alpha, dim=-1)                 # [b, L] 

    def _score_sentence(self, emissions: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        batch, seq_len, L, S = emissions.shape
        gold_emit = emissions.gather(-1, tags.unsqueeze(-1).clamp(0, S - 1)).squeeze(-1)
        score = self.start_transitions[torch.arange(L), tags[:, 0, :]].unsqueeze(0)
        score = score + gold_emit[:, 0, :]

        for t in range(1, seq_len):
            mask_t = mask[:, t].unsqueeze(-1)                # [b, 1]
            trans_score = self.transitions[
                torch.arange(L),
                tags[:, t - 1, :],
                tags[:, t, :],
            ]
            score = score + mask_t * (trans_score + gold_emit[:, t, :])

        last_idx = mask.long().sum(dim=1) - 1
        last_tags = tags[torch.arange(batch), last_idx, :]
        score = score + self.end_transitions[torch.arange(L), last_tags].unsqueeze(0)
        return score

    def neg_log_likelihood(self, emissions: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        log_Z    = self._forward_alg(emissions, mask)
        gold_score = self._score_sentence(emissions, tags, mask)
        nll = (log_Z - gold_score).mean()
        return nll

    def viterbi_decode(self, emissions: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        batch, seq_len, L, S = emissions.shape
        viterbi  = self.start_transitions.unsqueeze(0) + emissions[:, 0]
        backpointers = []

        for t in range(1, seq_len):
            emit_t = emissions[:, t].unsqueeze(-2)
            trans  = self.transitions.unsqueeze(0)
            score  = viterbi.unsqueeze(-1) + trans
            best_scores, best_states = score.max(dim=2)
            backpointers.append(best_states)

            mask_t = mask[:, t].view(batch, 1, 1)
            viterbi = torch.where(mask_t.bool(), best_scores + emit_t.squeeze(-2), viterbi)

        viterbi = viterbi + self.end_transitions.unsqueeze(0)
        _, best_last = viterbi.max(dim=-1)

        seq_tags = [best_last]
        for bp in reversed(backpointers):
            prev = seq_tags[-1]
            seq_tags.append(bp.gather(-1, prev.unsqueeze(-1)).squeeze(-1))

        seq_tags.reverse()
        return torch.stack(seq_tags, dim=1)

In [63]:
class _CRFOutput:
    def __init__(self, loss, logits):
        self.loss   = loss
        self.logits = logits

    _fields = ("loss", "logits")

    def __getitem__(self, key):
        if isinstance(key, str):
            return getattr(self, key)
        return (self.loss, self.logits)[key]

    def __len__(self):
        return 2

    def __iter__(self):
        yield self.loss
        yield self.logits

    def __contains__(self, key):
        return key in self._fields


class CRFTokenTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        out     = model(**inputs, labels=labels)
        loss    = out["loss"]
        wrapped = _CRFOutput(loss=loss, logits=out["logits"])
        return (loss, wrapped) if return_outputs else loss

    def create_optimizer(self):
        # Force standard AdamW — avoids fused/foreach CUDA optimizer bugs
        # with custom nn.Module models that the Trainer doesn't fully recognise
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.args.learning_rate,
            weight_decay=self.args.weight_decay,
        )
        return self.optimizer

    def _move_model_to_device(self, model, device):
        model.to(device)
        return model

    def _save_checkpoint(self, model, trial):
        for param in model.parameters():
            if not param.data.is_contiguous():
                param.data = param.data.contiguous()
        super()._save_checkpoint(model, trial)


In [64]:
class BiomedBERTWithCRF(nn.Module):
    """
    Replaces the raw sigmoid classifier with BERT + CRF structured decoding.
    """

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        dropout_prob: float = 0.1,
    ):
        super().__init__()
        self.num_labels = num_labels

        self.bert = AutoModel.from_pretrained(model_name)
        hidden = self.bert.config.hidden_size

        self.dropout    = nn.Dropout(dropout_prob)
        self.emission_head = nn.Linear(hidden, num_labels * 3)
        self.crf = BinaryCRFLayer(num_labels)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
    ):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden  = self.dropout(outputs.last_hidden_state)

        raw_emit = self.emission_head(hidden)
        b, T, _ = raw_emit.shape
        emissions = raw_emit.view(b, T, self.num_labels, 3)

        loss   = None
        if labels is not None:
            valid_mask = (labels[:, :, 0] != -1.0) & (attention_mask == 1)
            crf_tags = self._multihot_to_bio(labels, valid_mask)
            loss = self.crf.neg_log_likelihood(emissions, crf_tags, valid_mask)

        return {"loss": loss, "logits": emissions}

    @staticmethod
    def _multihot_to_bio(labels: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        b, T, L = labels.shape
        bio = torch.zeros(b, T, L, dtype=torch.long, device=labels.device)
        pos = (labels == 1.0)
        prev_pos = torch.zeros(b, L, dtype=torch.bool, device=labels.device)

        for t in range(T):
            is_valid = mask[:, t]
            cur_pos  = pos[:, t, :]
            is_B = cur_pos & ~prev_pos
            is_I = cur_pos & prev_pos

            bio[:, t, :] = torch.where(is_B, torch.ones_like(bio[:, t, :]),
                           torch.where(is_I, 2 * torch.ones_like(bio[:, t, :]),
                           bio[:, t, :]))
            prev_pos = torch.where(is_valid.unsqueeze(-1), cur_pos, prev_pos)

        return bio

In [65]:
model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

label2id =  {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

tokenizer = AutoTokenizer.from_pretrained(model_name)

data_collator = DataCollatorForMultiLabelTokenClassification(tokenizer)

train_data = AlignPubTator(
    parsed_data=train_results, 
    tokenizer=tokenizer, 
    label2id=label2id
)
val_data = AlignPubTator(
    parsed_data = dev_results,
    tokenizer = tokenizer,
    label2id=label2id
)
test_dataset = AlignPubTator(
    parsed_data=test_results,
    tokenizer=tokenizer,
    label2id=label2id
)

model = BiomedBERTWithCRF(model_name, num_labels=len(label_list))

if torch.cuda.is_available():
    device = torch.device("cuda") #Nvidia
elif torch.backends.mps.is_available():
    device = torch.device("mps") #Macbook
else:
    device = torch.device("cpu") #CPU

model.to(device)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17431.74it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BiomedBERTWithCRF(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise

In [66]:
def compute_pos_weight(train_results, label_list, label2id):
    """Count positive and negative occurrences per label across all tokens."""
    num_labels = len(label_list)
    pos_counts = torch.zeros(num_labels)
    neg_counts = torch.zeros(num_labels)
    
    for item in train_results:
        for token_tags in item['tags']:
            multi_hot = torch.zeros(num_labels)
            if token_tags != ['O']:
                for tag in token_tags:
                    if tag in label2id:
                        multi_hot[label2id[tag]] = 1.0
            pos_counts += multi_hot
            neg_counts += (1 - multi_hot)
    
    pos_weight = neg_counts / pos_counts.clamp(min=1)
    pos_weight = pos_weight.clamp(max=50.0)
    return pos_weight

pos_weight = compute_pos_weight(train_results, label_list, label2id)


In [53]:
training_args = TrainingArguments(
    output_dir="./BioMedBERT-CRF-Results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=1e-3,
    bf16=False,
    fp16=True,
)

trainer = CRFTokenTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics_crf,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [67]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,99.948236,19.462032,0.000000,0.000000
2,8.716428,6.286962,0.000000,0.000000


KeyboardInterrupt: 

In [69]:
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
NUM_LABELS = 127
REPO_ID = "Sbruuns/Biomedbert_CRF"
FILENAME = "model.safetensors"
weights_cache_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
best_model = BiomedBERTWithCRF(model_name=MODEL_NAME, num_labels=NUM_LABELS)

#best_model = BiomedBERTWithCRF(model_name=MODEL_NAME, num_labels=NUM_LABELS)
checkpoint_path = "./results/checkpoint-2970/model.safetensors"

state_dict = load_file(weights_cache_path)
best_model.load_state_dict(state_dict, strict=False)
best_model.to(device)
best_model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16271.25it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BiomedBERTWithCRF(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise

In [70]:
import torch
from torch.utils.data import DataLoader
import numpy as np

# 1. Clear the arrays completely to fix the size mismatch!
all_predictions = []
all_labels = []

best_model.eval()
best_model.to(device)

test_loader = DataLoader(test_dataset, batch_size=8, collate_fn=data_collator)

print("Running a clean inference pass over the test set...")
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        emissions = outputs["logits"] # Shape: [batch, seq_len, 127, 3]
        
        # Save the RAW 4D emissions and labels straight to our lists
        all_predictions.append(emissions.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

print("Clean inference pass complete!")

Running a clean inference pass over the test set...
Clean inference pass complete!


In [35]:
all_labels

[array([[[ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [-1., -1., -1., ..., -1., -1., -1.],
         ...,
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.]],
 
        [[ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         ...,
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.]],
 
        [[ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         ...,
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.]],
 
        ...,
 
        [[ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,  0.,  0., ...,  0.,  0.,  0.],
         [ 0.,

In [ ]:
# 1. Concatenate batches along the batch axis (axis 0)
final_predictions = np.concatenate(all_predictions, axis=0)
final_labels = np.concatenate(all_labels, axis=0)

# 2. Wrap them into the stub object for your metric function
class EvalPredWrapper:
    def __init__(self, predictions, label_ids):
        self.predictions = predictions
        self.label_ids = label_ids
    
    def __iter__(self):
        yield self.predictions
        yield self.label_ids

eval_pred_stub = EvalPredWrapper(predictions=final_predictions, label_ids=final_labels)

# 3. Calculate metrics natively!
test_metrics = compute_metrics_crf(eval_pred_stub)

# 4. Print your results
print("TEST SET RESULTS")
print(f"F1 Micro: {test_metrics['f1_micro']:.6f}")
print(f"F1 Macro: {test_metrics['f1_macro']:.6f}")



       TEST SET RESULTS       
F1 Micro: 0.520242
F1 Macro: 0.290724


In [72]:
import torch
import numpy as np

def extract_readable_predictions(dataset, model, tokenizer, id2label, device, num_samples=50):
    model.eval()
    model.to(device)
    
    analysis_records = []
    
    # Grab a few raw items from your dataset directly to inspect sentences clearly
    for idx in range(min(num_samples, len(dataset))):
        item = dataset[idx]
        
        # Format tensors for a single-forward pass
        input_ids = torch.tensor(item["input_ids"]).unsqueeze(0).to(device)
        attention_mask = torch.tensor(item["attention_mask"]).unsqueeze(0).to(device)
        raw_labels = np.array(item["labels"]) # [seq_len, 127]
        
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            emissions = outputs["logits"].squeeze(0).cpu().numpy() # [seq_len, 127, 3]
            
        # Decode states: argmax over the 3 BIO slots
        pred_states = np.argmax(emissions, axis=-1) # [seq_len, 127]
        
        # Convert tokens back to strings
        tokens = tokenizer.convert_ids_to_tokens(item["input_ids"])
        
        sentence_data = []
        for t_idx, token in enumerate(tokens):
            # Skip padding/special structural markers
            if token in [tokenizer.pad_token, tokenizer.cls_token, tokenizer.sep_token] or raw_labels[t_idx, 0] == -1.0:
                continue
                
            # Locate text labels for true ground annotations
            true_active_indices = np.where(raw_labels[t_idx] == 1.0)[0]
            true_tags = [id2label[i] for i in true_active_indices]
            
            # Locate text labels for what your model predicted (B or I states > 0)
            pred_active_indices = np.where(pred_states[t_idx] > 0)[0]
            pred_tags = [id2label[i] for i in pred_active_indices]
            
            sentence_data.append({
                "token": token.replace("Ġ", ""), # Clean up RoBERTa space markers if present
                "true": true_tags if true_tags else ["O"],
                "pred": pred_tags if pred_tags else ["O"]
            })
            
        analysis_records.append({
            "sample_index": idx,
            "text_stream": sentence_data
        })
        
    return analysis_records

# Generate a small evaluation corpus for qualitative analysis
qualitative_corpus = extract_readable_predictions(test_dataset, best_model, tokenizer, id2label, device, num_samples=30)

In [73]:
qualitative_corpus

[{'sample_index': 0,
  'text_stream': [{'token': 'non', 'true': ['T131'], 'pred': ['T109']},
   {'token': 'diet', 'true': ['T131'], 'pred': ['T109']},
   {'token': 'inhibits', 'true': ['T052'], 'pred': ['T052']},
   {'token': 'apoptosis', 'true': ['T043'], 'pred': ['T043']},
   {'token': 'induced', 'true': ['T169'], 'pred': ['T169']},
   {'token': 'in', 'true': ['O'], 'pred': ['O']},
   {'token': 'pc12', 'true': ['T025'], 'pred': ['T025']},
   {'token': 'cells', 'true': ['T025'], 'pred': ['T025']},
   {'token': 'non', 'true': ['T131'], 'pred': ['T109']},
   {'token': 'and', 'true': ['O'], 'pred': ['O']},
   {'token': 'short', 'true': ['T131'], 'pred': ['T109']},
   {'token': 'chain', 'true': ['T131'], 'pred': ['T109']},
   {'token': 'non', 'true': ['T131'], 'pred': ['T109']},
   {'token': 'eth', 'true': ['T131'], 'pred': ['T109']},
   {'token': 'such', 'true': ['O'], 'pred': ['O']},
   {'token': 'as', 'true': ['O'], 'pred': ['O']},
   {'token': 'np', 'true': ['T131'], 'pred': ['T109']}

In [ ]:
from transformers import AutoConfig
config = AutoConfig.from_pretrained(model_name)
config.id2label = {i: label for i, label in enumerate(label_list)} 
config.label2id = {label: i for i, label in enumerate(label_list)}
config.auto_map = {
    "AutoModelForTokenClassification": "modeling_biomedbert_crf.BiomedBERTWithCRF"
}
config.save_pretrained("./results/checkpoint-2970")

In [ ]:

# 1. Clear the arrays completely to fix the size mismatch!
all_predictions = []
all_labels = []

best_model.eval()
best_model.to(device)

test_loader = DataLoader(test_dataset, batch_size=8, collate_fn=data_collator)

print("Running a clean inference pass over the test set...")
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        emissions = outputs["logits"] # Shape: [batch, seq_len, 127, 3]
        
        # Save the RAW 4D emissions and labels straight to our lists
        all_predictions.append(emissions.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

print("Clean inference pass complete!")

# 1. Concatenate batches along the batch axis (axis 0)
final_predictions = np.concatenate(all_predictions, axis=0)
final_labels = np.concatenate(all_labels, axis=0)

# 2. Wrap them into the stub object for your metric function
class EvalPredWrapper:
    def __init__(self, predictions, label_ids):
        self.predictions = predictions
        self.label_ids = label_ids
    
    def __iter__(self):
        yield self.predictions
        yield self.label_ids

eval_pred_stub = EvalPredWrapper(predictions=final_predictions, label_ids=final_labels)

# 3. Calculate metrics natively!
test_metrics = compute_metrics_crf(eval_pred_stub)

# 4. Print your results
print("TEST SET RESULTS")

print(f"F1 Micro: {test_metrics['f1_micro']:.6f}")
print(f"F1 Macro: {test_metrics['f1_macro']:.6f}")


In [ ]:
def extract_readable_predictions_roberta(dataset, model, tokenizer, id2label, device, num_samples=30):
    model.eval()
    model.to(device)
    
    analysis_records = []
    
    # Grab items from your dataset directly to inspect sentences clearly
    for idx in range(min(num_samples, len(dataset))):
        item = dataset[idx]
        
        # Format tensors for a single-forward pass
        input_ids = torch.tensor(item["input_ids"]).unsqueeze(0).to(device)
        attention_mask = torch.tensor(item["attention_mask"]).unsqueeze(0).to(device)
        raw_labels = np.array(item["labels"]) # [seq_len, 127]
        
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            emissions = outputs["logits"] # [1, seq_len, 127, 3]
            
            # ====== VITERBI SEQUENCE DECODE ======
            # Runs global optimization pathways over transitions parameters.
            # Output Shape: [1, seq_len, 127] containing values 0, 1 (B), or 2 (I)
            pred_states_tensor = model.crf.viterbi_decode(emissions, attention_mask)
            pred_states = pred_states_tensor.squeeze(0).cpu().numpy() # [seq_len, 127]
            
        # Convert tokens back to strings (Uses RoBERTa BPE token standards)
        tokens = tokenizer.convert_ids_to_tokens(item["input_ids"])
        
        sentence_data = []
        for t_idx, token in enumerate(tokens):
            # Skip padding or special structural markers
            if token in [tokenizer.pad_token, tokenizer.cls_token, tokenizer.sep_token, "<s>", "</s>", "<pad>"] or raw_labels[t_idx, 0] == -1.0:
                continue
                
            # Locate text labels for true ground annotations
            true_active_indices = np.where(raw_labels[t_idx] == 1.0)[0]
            true_tags = [id2label[i] for i in true_active_indices]
            
            # Locate text labels for what your model predicted (B or I states > 0)
            pred_active_indices = np.where(pred_states[t_idx] > 0)[0]
            pred_tags = [id2label[i] for i in pred_active_indices]
            
            # Clean up RoBERTa BPE space marker ('Ġ') and sub-word markers ('##') if present
            clean_token = token.replace("Ġ", " ").replace("##", "")
            
            sentence_data.append({
                "token": clean_token.strip(), 
                "true": true_tags if true_tags else ["O"],
                "pred": pred_tags if pred_tags else ["O"]
            })
            
        analysis_records.append({
            "sample_index": idx,
            "text_stream": sentence_data
        })
        
    return analysis_records

# Generate evaluation corpus for qualitative analysis
qualitative_corpus = extract_readable_predictions_roberta(
    test_dataset, best_model, tokenizer, id2label, device, num_samples=30
)